# Baseline: Target-Only Regression

Validate baseline (target-only) performance before domain adaptation
- uses Graphormer embeddings
- evaluates every property with 5-fold CV
- 7 metrics (Pearson r, Spearman rho, R2, MAE, MSE, RMSE, MBE)

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = r'..'
DATA_DIR = os.path.join(ROOT, 'data', 'cv_datasets')
FEAT_DIR = os.path.join(ROOT, 'features')
RES_DIR = os.path.join(ROOT, 'results')
os.makedirs(RES_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROPERTIES = ['clearance', 'half_life', 'fu']
N_FOLDS = 5

print("="*100)
print("Baseline: Target-Only Regression (5-Fold CV)")
print("="*100)
print(f"Device: {DEVICE}")
print(f"Data: {DATA_DIR}")
print(f"Features: {FEAT_DIR}")
print(f"Results: {RES_DIR}\n")

# Shared training utilities: inner train/val split + val-based early stopping (test used only for final evaluation)
sys.path.insert(0, os.path.join(ROOT, '01_model'))
from train_utils import split_train_val, fit_early_stopping, target_smiles


Baseline: Target-Only Regression (5-Fold CV)
Device: cpu
Data: ..\data\cv_datasets
Features: ..\features
Results: ..\results



## Step 1: Load Graphormer Embeddings

In [2]:
print("Loading Graphormer embeddings...\n")

emb_path = os.path.join(FEAT_DIR, 'graphormer_cv_embeddings.npz')
emb_data = np.load(emb_path, allow_pickle=True)
emb_keys = emb_data['keys']
emb_X = emb_data['X']

smiles_to_emb = {smi: emb_X[i] for i, smi in enumerate(emb_keys)}

print(f"✓ Loaded {len(smiles_to_emb)} embeddings")
print(f"  Embedding dim: {emb_X.shape[1]}")
print(f"  Memory: {emb_X.nbytes / (1024**2):.1f} MB\n")

Loading Graphormer embeddings...

✓ Loaded 58304 embeddings
  Embedding dim: 768
  Memory: 170.8 MB



## Step 2: Define Model & Functions

In [3]:
class DirectRegressionModel(nn.Module):
    def __init__(self, d_in=768, d_hidden=256, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Linear(d_hidden // 2, 1)

    def forward(self, x):
        h = self.encoder(x)
        y = self.head(h).squeeze(-1)
        return y

print("✓ DirectRegressionModel defined")
model = DirectRegressionModel()
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

✓ DirectRegressionModel defined
  Parameters: 229,889


In [4]:
def load_fold_data(prop, fold_idx, split):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_target_{split}.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path)
    smiles = df['smiles'].astype(str).tolist()
    
    X_list = []
    y_list = []
    missing = 0
    
    for smi, y in zip(smiles, df['value'].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
        else:
            missing += 1
    
    if missing > 0:
        print(f"    Warning: {missing}/{len(smiles)} missing embeddings")
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    
    return X, y

def train_epoch(model, X_batch, y_batch, optimizer, criterion, device):
    model.train()
    perm = np.random.permutation(len(X_batch))
    total_loss = 0.0
    n_batches = 0

    for i in range(0, len(X_batch), 32):
        idx = perm[i:i + 32]
        x = torch.tensor(X_batch[idx], device=device, dtype=torch.float32)
        y = torch.tensor(y_batch[idx], device=device, dtype=torch.float32)

        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches

@torch.no_grad()
def evaluate(model, X_test, y_test, device):
    model.eval()
    x = torch.tensor(X_test, device=device, dtype=torch.float32)
    y_pred = model(x).cpu().numpy()
    y_true = y_test

    pearson_r = np.corrcoef(y_true, y_pred)[0, 1]
    spearman_rho, _ = spearmanr(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    mbe = np.mean(y_pred - y_true)

    return {
        'pearson_r': pearson_r,
        'spearman_rho': spearman_rho,
        'r2': r2,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mbe': mbe,
    }, y_pred

print("✓ All functions defined")

# Test data loading
print("\nTesting data loading...")
for prop in PROPERTIES:
    X_train, y_train = load_fold_data(prop, 0, 'train')
    X_test, y_test = load_fold_data(prop, 0, 'test')
    if X_train is not None:
        print(f"  {prop}: train={len(y_train)}, test={len(y_test)}")

✓ All functions defined

Testing data loading...
  clearance: train=821, test=206
  half_life: train=931, test=233
  fu: train=282, test=71


## Step 3: Train & Evaluate (5-Fold)

In [5]:
print("\n" + "="*100)
print("BASELINE EVALUATION: Target-Only Regression")
print("="*100 + "\n")

property_summaries = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    prop_results = []
    
    for fold_idx in range(N_FOLDS):
        print(f"\n  Fold {fold_idx}:")
        
        X_train, y_train = load_fold_data(prop, fold_idx, 'train')
        X_test, y_test = load_fold_data(prop, fold_idx, 'test')
        
        if X_train is None or len(X_train) == 0:
            print(f"    No training data")
            continue
        
        torch.manual_seed(fold_idx)
        np.random.seed(fold_idx)
        
        model = DirectRegressionModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        criterion = nn.HuberLoss(delta=1.0)
        
        # target train → scaffold-based inner train/val (early stopping on val R², test used only for final evaluation)
        X_train, y_train, X_val, y_val = split_train_val(X_train, y_train, fold_idx, smiles=target_smiles(DATA_DIR, prop, fold_idx, 'train', smiles_to_emb))
        
        def run_epoch():
            train_epoch(model, X_train, y_train, optimizer, criterion, DEVICE)
        
        best_epoch, best_val_r2 = fit_early_stopping(
            model, run_epoch, lambda m, X, y: evaluate(m, X, y, DEVICE)[0], X_val, y_val)
        print(f"    best epoch={best_epoch} | val r2={best_val_r2:.4f}")
        
        metrics, _ = evaluate(model, X_test, y_test, DEVICE)
        prop_results.append({
            'fold': fold_idx,
            'pearson_r': metrics['pearson_r'],
            'spearman_rho': metrics['spearman_rho'],
            'r2': metrics['r2'],
            'mae': metrics['mae'],
            'mse': metrics['mse'],
            'rmse': metrics['rmse'],
            'mbe': metrics['mbe'],
            'val_r2': best_val_r2,
            'best_epoch': best_epoch,
        })
        print(f"    r={metrics['pearson_r']:.4f}, rho={metrics['spearman_rho']:.4f}, r2={metrics['r2']:.4f}, mae={metrics['mae']:.4f}, rmse={metrics['rmse']:.4f}")
    
    if prop_results:
        prop_df = pd.DataFrame(prop_results)
        property_summaries[prop] = prop_df
        
        print(f"\n  {prop.upper()} SUMMARY (5-Fold Average)")
        print(f"    Pearson r:    {prop_df['pearson_r'].mean():.4f} +/- {prop_df['pearson_r'].std():.4f}")
        print(f"    Spearman rho: {prop_df['spearman_rho'].mean():.4f} +/- {prop_df['spearman_rho'].std():.4f}")
        print(f"    R2:           {prop_df['r2'].mean():.4f} +/- {prop_df['r2'].std():.4f}")
        print(f"    MAE:          {prop_df['mae'].mean():.4f} +/- {prop_df['mae'].std():.4f}")
        print(f"    MSE:          {prop_df['mse'].mean():.4f} +/- {prop_df['mse'].std():.4f}")
        print(f"    RMSE:         {prop_df['rmse'].mean():.4f} +/- {prop_df['rmse'].std():.4f}")
        print(f"    MBE:          {prop_df['mbe'].mean():.4f} +/- {prop_df['mbe'].std():.4f}")

print(f"\n{'='*80}")


BASELINE EVALUATION: Target-Only Regression


Property: CLEARANCE

  Fold 0:
    best epoch=17 | val r2=0.1722
    r=0.3130, rho=0.3147, r2=0.0842, mae=0.6923, rmse=0.9366

  Fold 1:
    best epoch=15 | val r2=0.1847
    r=0.2661, rho=0.2481, r2=0.0093, mae=0.7187, rmse=0.9854

  Fold 2:
    best epoch=23 | val r2=0.1103
    r=0.3587, rho=0.3625, r2=0.0582, mae=0.7035, rmse=0.8901

  Fold 3:
    best epoch=17 | val r2=0.1172
    r=0.4419, rho=0.4660, r2=0.1726, mae=0.6312, rmse=0.8531

  Fold 4:
    best epoch=8 | val r2=0.1260
    r=0.3534, rho=0.4082, r2=0.1210, mae=0.7870, rmse=1.0917

  CLEARANCE SUMMARY (5-Fold Average)
    Pearson r:    0.3466 +/- 0.0650
    Spearman rho: 0.3599 +/- 0.0839
    R2:           0.0891 +/- 0.0619
    MAE:          0.7065 +/- 0.0559
    MSE:          0.9120 +/- 0.1811
    RMSE:         0.9514 +/- 0.0928
    MBE:          0.0357 +/- 0.0773

Property: HALF_LIFE

  Fold 0:
    best epoch=4 | val r2=0.0681
    r=0.3985, rho=0.3985, r2=0.1540, mae=0.7468, 

## Step 4: Summary & Save

In [6]:
print("\n" + "="*100)
print("BASELINE SUMMARY - ALL METRICS")
print("="*100 + "\n")

for prop in PROPERTIES:
    if prop in property_summaries:
        df = property_summaries[prop]
        print(f"{prop.upper()}:")
        print(f"  Pearson r:    {df['pearson_r'].mean():.4f} +/- {df['pearson_r'].std():.4f}")
        print(f"  Spearman rho: {df['spearman_rho'].mean():.4f} +/- {df['spearman_rho'].std():.4f}")
        print(f"  R2:           {df['r2'].mean():.4f} +/- {df['r2'].std():.4f}")
        print(f"  MAE:          {df['mae'].mean():.4f} +/- {df['mae'].std():.4f}")
        print(f"  MSE:          {df['mse'].mean():.4f} +/- {df['mse'].std():.4f}")
        print(f"  RMSE:         {df['rmse'].mean():.4f} +/- {df['rmse'].std():.4f}")
        print(f"  MBE:          {df['mbe'].mean():.4f} +/- {df['mbe'].std():.4f}")
        print()

print("="*100)
print("Expected baseline: Spearman rho ~ 0.38")
print("Target-only baseline (no domain adaptation)")
print("="*100 + "\n")

for prop in PROPERTIES:
    if prop in property_summaries:
        out_path = os.path.join(RES_DIR, f'01_baseline_{prop}.csv')
        property_summaries[prop].to_csv(out_path, index=False)
        print(f"Saved: {out_path}")

print(f"\nAll results saved to: {RES_DIR}")


BASELINE SUMMARY - ALL METRICS

CLEARANCE:
  Pearson r:    0.3466 +/- 0.0650
  Spearman rho: 0.3599 +/- 0.0839
  R2:           0.0891 +/- 0.0619
  MAE:          0.7065 +/- 0.0559
  MSE:          0.9120 +/- 0.1811
  RMSE:         0.9514 +/- 0.0928
  MBE:          0.0357 +/- 0.0773

HALF_LIFE:
  Pearson r:    0.3900 +/- 0.0557
  Spearman rho: 0.4159 +/- 0.0421
  R2:           0.1317 +/- 0.0662
  MAE:          0.7098 +/- 0.0742
  MSE:          0.8630 +/- 0.1532
  RMSE:         0.9259 +/- 0.0840
  MBE:          0.0149 +/- 0.0999

FU:
  Pearson r:    0.5616 +/- 0.1103
  Spearman rho: 0.5807 +/- 0.1062
  R2:           0.2439 +/- 0.1641
  MAE:          0.6822 +/- 0.0767
  MSE:          0.7289 +/- 0.1977
  RMSE:         0.8481 +/- 0.1094
  MBE:          -0.0866 +/- 0.1655

Expected baseline: Spearman rho ~ 0.38
Target-only baseline (no domain adaptation)

Saved: ..\results\01_baseline_clearance.csv
Saved: ..\results\01_baseline_half_life.csv
Saved: ..\results\01_baseline_fu.csv

All results s